In [ ]:
%py
spark.catalog.setCurrentCatalog("purgo_databricks")

# PySpark script: Refactored Sales KPI Pipeline with Classes and Functions
# Purpose: Refactor sales KPI pipeline into reusable classes and functions for maintainability and testability
# Author: Giang Nguyen
# Date: 2025-10-13
# Description: This script loads product, sales, and market share data from Unity Catalog tables, computes KPIs (YoY growth, market penetration, sales rank) for each product per year, and writes results to the output table. It includes schema validation, error handling, and is structured for unit/integration testing.

# -- Required PySpark imports for data engineering
from pyspark.sql import DataFrame  
from pyspark.sql.functions import col, lit, when, sum as _sum, avg, round, year, row_number, lag  
from pyspark.sql.window import Window  
from pyspark.sql.types import StringType, IntegerType, DoubleType, DateType, StructType, StructField  

# -- SparkSession is already available in Databricks, do not initialize here
# from pyspark.sql import SparkSession  # built-in

# -- Set Spark configuration for shuffle partitions
spark.conf.set("spark.sql.shuffle.partitions", "200")

class TableSchemaValidator:
    """
    Validates that a DataFrame matches the expected schema.
    Args:
        expected_schema (StructType): The expected schema.
    Methods:
        validate(df: DataFrame, table_name: str) -> None
            Raises ValueError if schema does not match.
    """
    def __init__(self, expected_schema: StructType):
        self.expected_schema = expected_schema

    def validate(self, df: DataFrame, table_name: str) -> None:
        """
        Validates DataFrame schema against expected schema.
        Args:
            df (DataFrame): DataFrame to validate.
            table_name (str): Name of the table for error messages.
        Returns:
            None. Raises ValueError if schema mismatch.
        """
        actual_fields = {f.name: f.dataType for f in df.schema.fields}
        for field in self.expected_schema.fields:
            if field.name not in actual_fields:
                raise ValueError(f"Missing required column {field.name} in {table_name}")
            if type(actual_fields[field.name]) != type(field.dataType):
                raise TypeError(f"Type mismatch for {field.name} in {table_name}: expected {field.dataType}, got {actual_fields[field.name]}")
        # Check for duplicate primary keys if applicable
        if table_name == "product_data":
            if df.groupBy("product_id").count().filter(col("count") > 1).count() > 0:
                raise ValueError("Duplicate product_id found in product_data")
        if table_name == "product_sales_data":
            if df.groupBy("transaction_id").count().filter(col("count") > 1).count() > 0:
                raise ValueError("Duplicate transaction_id found in product_sales_data")

class ProductSalesKPIProcessor:
    """
    Processes product, sales, and market share data to compute KPIs.
    Args:
        product_df (DataFrame): Product data.
        sales_df (DataFrame): Sales data.
        market_share_df (DataFrame): Market share data.
    Methods:
        process() -> DataFrame
            Returns DataFrame with computed KPIs.
    """
    def __init__(self, product_df: DataFrame, sales_df: DataFrame, market_share_df: DataFrame):
        self.product_df = product_df
        self.sales_df = sales_df
        self.market_share_df = market_share_df

    def process(self) -> DataFrame:
        """
        Computes KPIs for products per year.
        Returns:
            DataFrame: Final KPI DataFrame.
        """
        # Filter out records with null sales_amount and add sales_year
        sales_df = self.sales_df.filter(col("sales_amount").isNotNull()) \
                                .withColumn("sales_year", year(col("sales_date")))

        # Join product info
        sales_enriched_df = sales_df.join(self.product_df, sales_df.sales_product_id == self.product_df.product_id, "left") \
                                    .drop(self.product_df.product_id)

        # Join market share info
        sales_market_df = sales_enriched_df.join(self.market_share_df, sales_enriched_df.sales_product_id == self.market_share_df.ms_product_id, "left") \
                                           .drop("ms_product_id")

        # Aggregate total sales and average market share
        agg_df = sales_market_df.groupBy("sales_product_id", "sales_year", "product_name", "market_segment") \
            .agg(
                _sum("sales_amount").cast(IntegerType()).alias("total_sales"),
                round(avg("market_share_pct"), 2).alias("avg_market_share")
            )

        # Window for previous year sales
        window_spec = Window.partitionBy("sales_product_id").orderBy("sales_year")
        agg_df = agg_df.withColumn("prev_year_sales", lag("total_sales", 1).over(window_spec))

        # Calculate YoY growth percentage
        agg_df = agg_df.withColumn("yoy_growth_pct",
                                  round(((col("total_sales") - col("prev_year_sales")) / col("prev_year_sales")) * 100, 2))

        # Market penetration flag
        agg_df = agg_df.withColumn("market_penetration_flag",
                                  when(col("avg_market_share") > 25, lit("High"))
                                  .when((col("avg_market_share") <= 25) & (col("avg_market_share") >= 10), lit("Medium"))
                                  .otherwise(lit("Low")))

        # Rank products by total sales per year
        rank_window = Window.partitionBy("sales_year").orderBy(col("total_sales").desc())
        agg_df = agg_df.withColumn("sales_rank", row_number().over(rank_window))

        # Final output columns
        final_df = agg_df.select(
            "sales_year", "sales_product_id", "product_name", "market_segment",
            "total_sales", "prev_year_sales", "yoy_growth_pct",
            "avg_market_share", "market_penetration_flag", "sales_rank"
        )
        return final_df

class OutputTableWriter:
    """
    Writes the final KPI DataFrame to the output table.
    Args:
        output_table (str): Fully qualified table name.
    Methods:
        write(df: DataFrame) -> None
            Writes DataFrame to output table in overwrite mode.
    """
    def __init__(self, output_table: str):
        self.output_table = output_table

    def write(self, df: DataFrame) -> None:
        """
        Writes DataFrame to output table.
        Args:
            df (DataFrame): DataFrame to write.
        Returns:
            None. Raises Exception if write fails or schema mismatch.
        """
        # Validate output schema before writing
        expected_schema = StructType([
            StructField("sales_year", IntegerType(), True),
            StructField("sales_product_id", StringType(), True),
            StructField("product_name", StringType(), True),
            StructField("market_segment", StringType(), True),
            StructField("total_sales", IntegerType(), True),
            StructField("prev_year_sales", IntegerType(), True),
            StructField("yoy_growth_pct", DoubleType(), True),
            StructField("avg_market_share", DoubleType(), True),
            StructField("market_penetration_flag", StringType(), True),
            StructField("sales_rank", IntegerType(), True)
        ])
        actual_fields = {f.name: f.dataType for f in df.schema.fields}
        for field in expected_schema.fields:
            if field.name not in actual_fields:
                raise ValueError(f"Output table sales_kpi schema mismatch: missing {field.name}")
            if type(actual_fields[field.name]) != type(field.dataType):
                raise TypeError(f"Output table sales_kpi schema mismatch: {field.name} expected {field.dataType}, got {actual_fields[field.name]}")
        try:
            df.write.mode("overwrite").saveAsTable(self.output_table)
        except Exception as e:
            raise RuntimeError(f"Failed to write to output table sales_kpi: {str(e)}")

def load_table(table_name: str, expected_schema: StructType) -> DataFrame:
    """
    Loads a table from Unity Catalog and validates its schema.
    Args:
        table_name (str): Fully qualified table name.
        expected_schema (StructType): Expected schema for validation.
    Returns:
        DataFrame: Loaded DataFrame.
    Raises:
        Exception if file/table is missing or schema is invalid.
    """
    try:
        df = spark.read.table(table_name)
    except Exception as e:
        raise RuntimeError(f"Failed to open table {table_name}: {str(e)}")
    TableSchemaValidator(expected_schema).validate(df, table_name.split(".")[-1])
    return df

def run_sales_kpi_pipeline():
    """
    Orchestrates the sales KPI pipeline: loads data, processes KPIs, writes output.
    Returns:
        None
    """
    # Define expected schemas for input tables
    product_schema = StructType([
        StructField("product_id", StringType(), True),
        StructField("product_name", StringType(), True),
        StructField("market_segment", StringType(), True)
    ])
    sales_schema = StructType([
        StructField("transaction_id", StringType(), True),
        StructField("sales_product_id", StringType(), True),
        StructField("sales_amount", IntegerType(), True),
        StructField("sales_date", DateType(), True)
    ])
    market_share_schema = StructType([
        StructField("ms_product_id", StringType(), True),
        StructField("market_share_pct", DoubleType(), True)
    ])
    # Load tables with schema validation and error handling
    product_df = load_table("purgo_playground.product_data", product_schema)
    sales_df = load_table("purgo_playground.product_sales_data", sales_schema)
    market_share_df = load_table("purgo_playground.product_marketshare_data", market_share_schema)
    # Process KPIs
    processor = ProductSalesKPIProcessor(product_df, sales_df, market_share_df)
    final_df = processor.process()
    # Write output
    writer = OutputTableWriter("purgo_playground.sales_kpi")
    writer.write(final_df)

# -- Run the pipeline
run_sales_kpi_pipeline()

# -- Do not call spark.stop() in Databricks
# spark.stop()
